# Predicting the Duration of a Taxi Ride
In this notebook, we will create a machine learning model to predict the duration of a taxi ride based on different parameters.

In [ ]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [ ]:
df = pd.read_parquet("../data/green_taxi/green_tripdata_2021-01.parquet")
df

`trip_type` represents how the taxi was called. If it was hailed from the street, the value of this column is 1. If it was dispatched (possibly after a call), the column value is 2.

## Compute duration
We compute the duration of each ride based on the start and end time of a trip.

In [ ]:
df.loc[:, "lpep_pickup_datetime"] = pd.to_datetime(
    df["lpep_pickup_datetime"], format="%Y-%m-%d %H:%M:%S"
)
df.loc[:, "lpep_dropoff_datetime"] = pd.to_datetime(
    df["lpep_dropoff_datetime"], format="%Y-%m-%d %H:%M:%S"
)
df.loc[:, "duration"] = (
    df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
).dt.total_seconds() / 60
df = df.reset_index(drop=True)
df

In [ ]:
df["duration"].describe(percentiles=[0.01, 0.02, 0.25, 0.5, 0.75, 0.98, 0.99])

Based on the description of ride durations, we can make the following observations:
- Some trips are too long (almost an entire day) while others are too short (less than a minute).
- 98% of trips have a duration less than 1 hour which sounds reasonable.
- About 2% of the trips have a duration of less than 1 minute.

Given these observations, we filter the data for trips greater than 1 minute and less than an hour.

In [ ]:
df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
df

## Feature selection
We define numeric and categorical features that act as predictors in our model. To begin with, we choose the pickup location ID, dropoff location ID, and trip distance as the features.

In [ ]:
categorical_vars = ["PULocationID", "DOLocationID"]
numerical_vars = ["trip_distance"]

We encode the categorical variables as one-hot vectors before using them as model features.

In [ ]:
# Convert the location IDs from integers to strings so that they can
# be converted to one-hot encoded features by DictVectorizer.
df.loc[:, categorical_vars] = df[categorical_vars].astype(str)

In [ ]:
# We pass all features to DictVectorizer, including the numerical ones as
# it skips numerical features by default.
train_dict = df[categorical_vars + numerical_vars].to_dict(orient="records")

In [ ]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dict)
X_train

We print the features names created by CV to ensure that the one-hot encoding is complete. For the categorical variables, the feature names should include separate features for each category.

In [ ]:
dv.feature_names_

In [ ]:
target = "duration"
y_train = df[target].values
y_train

## Model fit
Next, we fit a linear regression model to the data.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

## Visualise model predictions
We now compare the distribution of actual durations against predicted durations from the linear regression model.

In [ ]:
y_pred = lr.predict(X_train)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    y_train,
    color="blue",
    label="Actual duration",
    kde=True,
    stat="density",
    bins=50,
    alpha=0.5,
)
sns.histplot(
    y_pred,
    color="red",
    label="Predicted duration",
    kde=True,
    stat="density",
    bins=50,
    alpha=0.5,
)
plt.xlabel("Duration (minutes)")
plt.ylabel("Density")
plt.legend()
plt.title("Distribution of Actual vs Predicted Ride Duration")
plt.show()

## Metrics
Next, we compute the error between the actuals and the predictions. Since it's a regression problem, we use mean absolute error as the baseline.

In [ ]:
mean_absolute_error(y_train, y_pred)

In [ ]:
mean_absolute_percentage_error(y_train, y_pred)

The error indicates that our model currently gets the duration prediction wrong by about 7 minutes which is equivalent to a 72% deviation from the actual duration. This is a large deviation and indicates that we need to find a better solution.

A problem with our existing approach is that we use the entire available data for training and then re-use it for inference. The best practice is to split the data into training and test sets so that the model can generalise better and perform well on data it has not seen before. In order to do so, we first refactor the code to make it easier to execute for multiple data sources and dataframes.

## Code refactor
First, we collect the data preprocessing steps into a single function.

In [ ]:
def load_and_process_dataframe(filename: str) -> pd.DataFrame:
    """
    Loads a parquet file containing taxi trip data, processes datetime columns,
    computes trip duration in minutes, filters out trips with duration < 1 or > 60 minutes,
    and converts categorical location ID columns to string type for one-hot encoding.

    Parameters
    ----------
    filename : str
        Path to the parquet file to load.

    Returns
    -------
    pd.DataFrame
        Processed DataFrame ready for feature extraction and modeling.
    """
    # Load the parquet file into a DataFrame
    df = pd.read_parquet(filename)

    # Convert pickup and dropoff columns to datetime
    df.loc[:, "lpep_pickup_datetime"] = pd.to_datetime(
        df["lpep_pickup_datetime"], format="%Y-%m-%d %H:%M:%S"
    )
    df.loc[:, "lpep_dropoff_datetime"] = pd.to_datetime(
        df["lpep_dropoff_datetime"], format="%Y-%m-%d %H:%M:%S"
    )

    # Compute trip duration in minutes
    df.loc[:, "duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    # Reset index after filtering
    df = df.reset_index(drop=True)

    # Filter trips to keep only those with reasonable durations
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]

    # Convert location ID columns to string for one-hot encoding
    categorical_vars = ["PULocationID", "DOLocationID"]
    df.loc[:, categorical_vars] = df[categorical_vars].astype(str)

    return df

We now load the trip data for January 2021 to train the model and then use the data from February 2021 to evaluate the model.

In [ ]:
df_train = load_and_process_dataframe("../data/green_taxi/green_tripdata_2021-01.parquet")
df_test = load_and_process_dataframe("../data/green_taxi/green_tripdata_2021-02.parquet")

We now convert the categorical variables to one-hot encodings. In this transformation, we take care to fit the vectoriser only to the training data and use these fits to transform both the training and test data.

In [ ]:
dv = DictVectorizer()
train_dicts = df_train[categorical_vars + numerical_vars].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)

test_dicts = df_test[categorical_vars + numerical_vars].to_dict(orient="records")
X_test = dv.transform(test_dicts)

In [ ]:
target = "duration"
y_train = df_train[target].values
y_test = df_test[target].values

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

mean_absolute_error(y_test, y_pred), mean_absolute_percentage_error(y_test, y_pred)

In [ ]:
# Save the model and DictVectorizer to disk
with open("../models/green_taxi_lin_reg.bin", "wb") as f_out:
    pickle.dump((dv, lr), f_out)

Based on the metrics, we can observe that the model performs slightly poorly on the test set (7.6 minutes compared to 7.2 minutes on the training set). The mean absolute percentage error is surprisingly lower on the test set (66% compared to 72% on the training set). Regardless, the metrics indicate that the model is a poor fit to the data. 

To derive a model that fits the data better, we can:
- Experiment with alternative linear regression models that include regularisation like Lasso or Ridge.
- Include additional features, derive new features from existing features, or remoe existing features.  